# QA: Phase Error Correction Quality

Inspect downsampled (128×128×64) velocity data for every patient in the splits file.
For each patient, a mid-axial slice is shown at a single cardiac frame:

| Row | Content | Colormap |
|-----|---------|----------|
| 1   | CNN raw correction (per-timepoint network output) — vx, vy, vz | jet |
| 2   | CNN polyfit correction (time-independent) — vx, vy, vz | jet |
| 3   | Raw manual correction (corrected vel − uncorrected vel) — vx, vy, vz | jet |
| 4   | Manual polyfit correction (time-independent) — vx, vy, vz | jet |
| 5   | Manual polyfit − CNN polyfit on mediastinum — vx, vy, vz | jet |
| 6   | CNN corrected velocity (uncorr + polyfit) — vx, vy, vz | RdBu_r |
| 7   | CNN improvement: \|GT−uncorr\| − \|GT−CNN\| — vx, vy, vz (unmasked) | RdBu_r |
| 8   | Manually corrected velocity (piecewise) — vx, vy, vz | RdBu_r |
| 9   | Manually polyfit corrected velocity (uncorr + man. polyfit) — vx, vy, vz | RdBu_r |
| 10  | Uncorrected velocity — vx, vy, vz | RdBu_r |
| 11  | Magnitude, tissue mask, used pixels | gray / RdBu_r |

Rows 1–4 show full-FOV correction fields (no tissue masking).
Row 5 shows the polyfit correction residual on mediastinum (body) pixels.
Rows 6–8 are masked to tissue regions.
Rows 1–2, 5 require inference results.

Images are saved to `./qa_pec_images/`.

In [15]:
import platform
from pathlib import Path

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import zoom

from vascular_superenhancement.utils.path_config import load_path_config, _PROJECT_ROOT

config_name = "local_mac" if platform.system() == "Darwin" else "all_patients"
pc = load_path_config(config_name)

WORKING_DIR = pc.working_dir
PATIENT_DATA_DIR = WORKING_DIR / "patient_data"
REPO_ROOT = _PROJECT_ROOT
DOWNSAMPLED_FOLDER = "downsampled_full_fov_128x128x64"

RUN_NAME = "lucky-eon_epoch_68"
INFERENCE_DIR = WORKING_DIR.parent / "inference" / RUN_NAME

OUTPUT_DIR = Path("qa_pec_images") / RUN_NAME
OUTPUT_DIR.mkdir(exist_ok=True)

FRAME_INDEX = 4

print(f"Patient data dir: {PATIENT_DATA_DIR}")
print(f"Inference dir:    {INFERENCE_DIR}")
print(f"Output dir:       {OUTPUT_DIR.resolve()}")

Patient data dir: /Users/yakhilesh/Files/7_PhD/vascular-superenhancement/code-base/vascular-superenhancement-4d-flow/working_dir/all_patients/patient_data
Inference dir:    /Users/yakhilesh/Files/7_PhD/vascular-superenhancement/code-base/vascular-superenhancement-4d-flow/working_dir/inference/lucky-eon_epoch_68
Output dir:       /Users/yakhilesh/Files/7_PhD/vascular-superenhancement/code-base/vascular-superenhancement-4d-flow/notebooks/data-qa/qa_pec_images/lucky-eon_epoch_68


In [16]:
splits_df = pd.read_csv(REPO_ROOT / "splits" / "splits_01-15-26.csv")

# --- Filtering options (set to None / empty to disable) ---
# Restrict to specific splits, e.g. ["test"] or ["train", "validation"]
FILTER_SPLITS: list[str] | None = None          # None = all splits
# Restrict to specific patient IDs, e.g. ["Biswifo", "Balboloop"]
FILTER_PATIENTS: list[str] | None = ["Biswifo"]   # None = all patients

patients_df = splits_df[splits_df["split"].isin(["train", "validation", "test"])].copy()
if FILTER_SPLITS:
    patients_df = patients_df[patients_df["split"].isin(FILTER_SPLITS)]
if FILTER_PATIENTS:
    patients_df = patients_df[patients_df["patient_id"].isin(FILTER_PATIENTS)]
patients_df = patients_df.sort_values(["split", "patient_id"]).reset_index(drop=True)

print(f"Total patients to inspect: {len(patients_df)}")
print(patients_df["split"].value_counts())
if FILTER_SPLITS:
    print(f"  (filtered to splits: {FILTER_SPLITS})")
if FILTER_PATIENTS:
    print(f"  (filtered to patients: {FILTER_PATIENTS})")

Total patients to inspect: 1
split
test    1
Name: count, dtype: int64
  (filtered to patients: ['Biswifo'])


In [17]:
COMPONENTS = ["vx", "vy", "vz"]

def load_axial_slice(nifti_path: Path, z_idx: int | None = None) -> np.ndarray:
    """Load a NIfTI file and return an axial slice as a 2-D array.

    If z_idx is None, the mid-axial slice is used.
    """
    vol = nib.load(str(nifti_path)).get_fdata(dtype=np.float32)
    if z_idx is None:
        z_idx = vol.shape[2] // 2
    return vol[:, :, z_idx]


def load_air_mask_slice(patient_dir: Path, pid: str, ds_shape: tuple,
                        z_idx: int | None = None) -> np.ndarray:
    """Load the correction air mask and return a resampled axial slice.

    The mask lives at original resolution in velocity_correction/; it is
    resampled to the downsampled grid so it can be used with downsampled data.
    """
    mask_path = patient_dir / "nifti" / "velocity_correction" / f"correction_air_mask_{pid}.nii.gz"
    mask_vol = nib.load(str(mask_path)).get_fdata(dtype=np.float32)

    # Resample to downsampled resolution if shapes differ
    if mask_vol.shape != ds_shape:
        scale = np.array(ds_shape) / np.array(mask_vol.shape)
        mask_vol = zoom(mask_vol, scale, order=0)  # nearest-neighbour for binary mask

    if z_idx is None:
        z_idx = mask_vol.shape[2] // 2
    return mask_vol[:, :, z_idx] > 0.5


def apply_mask(arr: np.ndarray, mask: np.ndarray) -> np.ma.MaskedArray:
    """Return a masked array where air pixels are masked out."""
    return np.ma.masked_where(~mask, arr)


def _try_load_axial_slice(nifti_path: Path, z_idx: int | None = None) -> np.ndarray | None:
    """Load an axial slice if the file exists, otherwise return None."""
    if not nifti_path.exists():
        return None
    return load_axial_slice(nifti_path, z_idx)


def make_qa_figure(pid: str, split: str, frame: int, ds_root: Path,
                   patient_dir: Path,
                   inf_root: Path | None = None,
                   z_idx: int | None = None) -> plt.Figure:
    """Generate a QA figure.

    Rows: CNN raw | CNN polyfit | Manual fit | CNN corr vel |
          Man. corr vel | Uncorr vel | Mag / Mask / Used Pixels.
    """
    mag_path = ds_root / "4d_flow_mag" / f"4d_flow_mag_{pid}_frame_{frame:02d}.nii.gz"
    mag_slice = load_axial_slice(mag_path, z_idx)

    uncorr = {}
    corr = {}
    for comp in COMPONENTS:
        uncorr[comp] = _try_load_axial_slice(
            ds_root / f"4d_flow_{comp}" / f"4d_flow_{comp}_{pid}_frame_{frame:02d}.nii.gz", z_idx
        )
        corr[comp] = _try_load_axial_slice(
            ds_root / f"4d_flow_{comp}_corr" / f"4d_flow_{comp}_corr_{pid}_frame_{frame:02d}.nii.gz", z_idx
        )

    has_uncorr = all(uncorr[c] is not None for c in COMPONENTS)
    has_corr = all(corr[c] is not None for c in COMPONENTS)
    has_diff = has_uncorr and has_corr
    diff = {}
    if has_diff:
        diff = {comp: corr[comp] - uncorr[comp] for comp in COMPONENTS}

    pred_raw = {}
    pred_poly = {}
    if inf_root is not None and inf_root.exists():
        poly_z = None
        if z_idx is not None:
            first_poly = inf_root / "predicted_corrected_velocity" / f"pred_correction_vx_{pid}.nii.gz"
            if first_poly.exists():
                poly_nz = nib.load(str(first_poly)).shape[2]
                ds_nz = nib.load(str(mag_path)).shape[2]
                poly_z = int(round(z_idx / (ds_nz - 1) * (poly_nz - 1)))

        for comp in COMPONENTS:
            pred_raw[comp] = _try_load_axial_slice(
                inf_root / "raw_predictions" / f"pred_correction_{comp}_t{frame:02d}.nii.gz", z_idx
            )
            pred_poly[comp] = _try_load_axial_slice(
                inf_root / "predicted_corrected_velocity" / f"pred_correction_{comp}_{pid}.nii.gz", poly_z
            )
    has_pred_raw = all(pred_raw.get(c) is not None for c in COMPONENTS)
    has_pred_poly = all(pred_poly.get(c) is not None for c in COMPONENTS)

    # De-normalise CNN predictions: stored as v/VENC, multiply back by VENC
    if (has_pred_raw or has_pred_poly) and inf_root is not None:
        coeff_path = inf_root / "predicted_corrected_velocity" / f"pred_poly_coefficients_{pid}.npz"
        if coeff_path.exists():
            venc = float(np.load(str(coeff_path))["venc"])
            for c in COMPONENTS:
                if pred_raw.get(c) is not None:
                    pred_raw[c] = pred_raw[c] * venc
                if pred_poly.get(c) is not None:
                    pred_poly[c] = pred_poly[c] * venc

    # CNN corrected = uncorrected + polyfit correction (resized if needed)
    cnn_corr = {}
    has_cnn_corr = has_uncorr and has_pred_poly
    if has_cnn_corr:
        for comp in COMPONENTS:
            poly_slice = pred_poly[comp]
            ref_shape = uncorr[comp].shape
            if poly_slice.shape != ref_shape:
                poly_slice = zoom(poly_slice,
                                  np.array(ref_shape) / np.array(poly_slice.shape),
                                  order=1)
            cnn_corr[comp] = uncorr[comp] + poly_slice

    # Manual polyfit corrected = uncorrected + manual polyfit correction
    man_poly_corr = {}
    has_man_poly_corr = False  # set after manual_poly is loaded below

    ds_shape = nib.load(str(mag_path)).shape
    tissue = load_air_mask_slice(patient_dir, pid, ds_shape, z_idx)
    mediastinum = tissue

    # Manual polyfit correction (time-independent, like CNN polyfit)
    # Stored normalized as v/VENC — de-normalise with VENC from patient data.
    # Files are at original resolution — resample to the downsampled grid.
    manual_poly = {}
    corr_dir = patient_dir / "nifti" / "velocity_correction"
    ref_shape_2d = mag_slice.shape
    man_venc_path = corr_dir / f"poly_coefficients_{pid}.npz"
    man_venc = float(np.load(str(man_venc_path))["venc"]) if man_venc_path.exists() else 1.0
    for comp in COMPONENTS:
        mp_path = corr_dir / f"ground_truth_correction_{comp}_{pid}.nii.gz"
        if mp_path.exists():
            mp_slice = load_axial_slice(mp_path, z_idx)
            if mp_slice.shape != ref_shape_2d:
                mp_slice = zoom(mp_slice,
                                np.array(ref_shape_2d) / np.array(mp_slice.shape),
                                order=1)
            manual_poly[comp] = mp_slice * man_venc
        else:
            manual_poly[comp] = None
    has_manual_poly = all(manual_poly[c] is not None for c in COMPONENTS)

    # Manual polyfit corrected velocity = uncorrected + manual polyfit
    has_man_poly_corr = has_uncorr and has_manual_poly
    if has_man_poly_corr:
        for comp in COMPONENTS:
            man_poly_corr[comp] = uncorr[comp] + manual_poly[comp]

    # Residual: manual polyfit − CNN polyfit on mediastinum (body pixels)
    corr_residual = {}
    has_corr_residual = has_manual_poly and has_pred_poly
    if has_corr_residual:
        for comp in COMPONENTS:
            cnn_slice = pred_poly[comp]
            man_slice = manual_poly[comp]
            ref_shape = man_slice.shape
            if cnn_slice.shape != ref_shape:
                cnn_slice = zoom(cnn_slice,
                                 np.array(ref_shape) / np.array(cnn_slice.shape),
                                 order=1)
            corr_residual[comp] = man_slice - cnn_slice

    vel_max = 300.0

    # Shared correction scale across CNN and manual correction rows
    corr_max = 1.0
    corr_vals = []
    if has_pred_raw:
        for c in COMPONENTS:
            corr_vals.append(pred_raw[c].ravel())
    if has_pred_poly:
        for c in COMPONENTS:
            corr_vals.append(pred_poly[c].ravel())
    if has_diff:
        for c in COMPONENTS:
            corr_vals.append(diff[c].ravel())
    if has_corr_residual:
        for c in COMPONENTS:
            corr_vals.append(corr_residual[c].ravel())
    if corr_vals:
        corr_max = np.percentile(np.abs(np.concatenate(corr_vals)), 99)
        if corr_max == 0:
            corr_max = 1.0

    bg_color = "0.25"
    vel_cmap = plt.cm.RdBu_r.copy()
    vel_cmap.set_bad(bg_color)
    jet_cmap = plt.cm.jet.copy()
    jet_cmap.set_bad(bg_color)
    resid_cmap = plt.cm.RdBu_r.copy()
    resid_cmap.set_bad(bg_color)

    z_label = f"z={z_idx}" if z_idx is not None else "z=mid"

    # CNN improvement: |GT - uncorrected| - |GT - CNN corrected|, unmasked
    cnn_improvement = {}
    has_cnn_improvement = has_corr and has_cnn_corr and has_uncorr
    if has_cnn_improvement:
        for comp in COMPONENTS:
            cnn_improvement[comp] = np.abs(corr[comp] - uncorr[comp]) - np.abs(corr[comp] - cnn_corr[comp])

    n_rows = 11
    fig, axes = plt.subplots(n_rows, 3, figsize=(14, n_rows * 4), constrained_layout=True)
    fig.suptitle(f"{pid}  [{split}]  frame {frame:02d}  {z_label}", fontsize=16, fontweight="bold")

    # Row 0: CNN raw correction (per-timepoint) — jet, full FOV
    for j, comp in enumerate(COMPONENTS):
        if has_pred_raw:
            im = axes[0, j].imshow(
                pred_raw[comp].T,
                origin="upper", cmap=jet_cmap, vmin=-corr_max, vmax=corr_max,
            )
            axes[0, j].set_title(f"CNN {comp}")
        else:
            axes[0, j].text(0.5, 0.5, f"CNN {comp}\nnot available",
                            transform=axes[0, j].transAxes,
                            ha="center", va="center", fontsize=12, color="white")
            axes[0, j].set_title(f"CNN {comp} (missing)")
    if has_pred_raw:
        fig.colorbar(im, ax=axes[0, :].tolist(), fraction=0.02, pad=0.02, label="correction")

    # Row 1: CNN polyfit correction (time-independent) — jet, full FOV
    for j, comp in enumerate(COMPONENTS):
        if has_pred_poly:
            im = axes[1, j].imshow(
                pred_poly[comp].T,
                origin="upper", cmap=jet_cmap, vmin=-corr_max, vmax=corr_max,
            )
            axes[1, j].set_title(f"CNN Polyfit {comp}")
        else:
            axes[1, j].text(0.5, 0.5, f"CNN Polyfit {comp}\nnot available",
                            transform=axes[1, j].transAxes,
                            ha="center", va="center", fontsize=12, color="white")
            axes[1, j].set_title(f"CNN Polyfit {comp} (missing)")
    if has_pred_poly:
        fig.colorbar(im, ax=axes[1, :].tolist(), fraction=0.02, pad=0.02, label="correction")

    # Row 2: Raw manual correction (corrected − uncorrected) — jet, full FOV
    for j, comp in enumerate(COMPONENTS):
        if has_diff:
            im = axes[2, j].imshow(
                diff[comp].T,
                origin="upper", cmap=jet_cmap, vmin=-corr_max, vmax=corr_max,
            )
            axes[2, j].set_title(f"Man. Corr−Uncorr {comp}")
        else:
            axes[2, j].text(0.5, 0.5, f"Man. Corr−Uncorr {comp}\nnot available",
                            transform=axes[2, j].transAxes,
                            ha="center", va="center", fontsize=12, color="white")
            axes[2, j].set_title(f"Man. Corr−Uncorr {comp} (missing)")
    if has_diff:
        fig.colorbar(im, ax=axes[2, :].tolist(), fraction=0.02, pad=0.02, label="correction")

    # Row 3: Manual polyfit correction — jet, full FOV
    for j, comp in enumerate(COMPONENTS):
        ax = axes[3, j]
        if has_manual_poly:
            im = ax.imshow(
                manual_poly[comp].T,
                origin="upper", cmap=jet_cmap, vmin=-corr_max, vmax=corr_max,
            )
            ax.set_title(f"Man. Polyfit {comp}")
        else:
            ax.text(0.5, 0.5, f"Man. Polyfit {comp}\nnot available",
                    transform=ax.transAxes,
                    ha="center", va="center", fontsize=12, color="white")
            ax.set_title(f"Man. Polyfit {comp} (missing)")
    if has_manual_poly:
        fig.colorbar(im, ax=axes[3, :].tolist(), fraction=0.02, pad=0.02, label="correction")

    # Row 4: Manual polyfit − CNN polyfit on mediastinum — RdBu_r
    for j, comp in enumerate(COMPONENTS):
        ax = axes[4, j]
        if has_corr_residual:
            im = ax.imshow(
                apply_mask(corr_residual[comp], mediastinum).T,
                origin="upper", cmap=resid_cmap, vmin=-corr_max, vmax=corr_max,
            )
            ax.set_title(f"Man.Poly−CNN Poly {comp}")
        else:
            ax.text(0.5, 0.5, f"Man.Poly−CNN Poly {comp}\nnot available",
                    transform=ax.transAxes,
                    ha="center", va="center", fontsize=10, color="white")
            ax.set_title(f"Man.Poly−CNN Poly {comp} (missing)")
    if has_corr_residual:
        fig.colorbar(im, ax=axes[4, :].tolist(), fraction=0.02, pad=0.02, label="polyfit residual")

    # Row 5: CNN corrected velocity (uncorr + polyfit) — RdBu_r, tissue masked
    for j, comp in enumerate(COMPONENTS):
        if has_cnn_corr:
            im = axes[5, j].imshow(
                apply_mask(cnn_corr[comp], tissue).T,
                origin="upper", cmap=vel_cmap, vmin=-vel_max, vmax=vel_max,
            )
            axes[5, j].set_title(f"CNN Corr. {comp}")
        else:
            axes[5, j].text(0.5, 0.5, f"CNN Corr. {comp}\nnot available",
                            transform=axes[5, j].transAxes,
                            ha="center", va="center", fontsize=12, color="white")
            axes[5, j].set_title(f"CNN Corr. {comp} (missing)")
    if has_cnn_corr:
        fig.colorbar(im, ax=axes[5, :].tolist(), fraction=0.02, pad=0.02, label="velocity")

    # Row 6: CNN improvement |GT−uncorr| − |GT−CNN| — RdBu_r, unmasked full FOV
    # Positive = CNN closer to GT (helped), Negative = uncorrected closer (CNN hurt)
    for j, comp in enumerate(COMPONENTS):
        ax = axes[6, j]
        if has_cnn_improvement:
            improv_max = np.percentile(np.abs(cnn_improvement[comp]), 99)
            if improv_max == 0:
                improv_max = 1.0
            im = ax.imshow(
                cnn_improvement[comp].T,
                origin="upper", cmap=resid_cmap, vmin=-improv_max, vmax=improv_max,
            )
            ax.set_title(f"|GT−uncorr|−|GT−CNN| {comp}")
        else:
            ax.text(0.5, 0.5, f"CNN improvement {comp}\nnot available",
                    transform=ax.transAxes,
                    ha="center", va="center", fontsize=10, color="white")
            ax.set_title(f"CNN improvement {comp} (missing)")
    if has_cnn_improvement:
        fig.colorbar(im, ax=axes[6, :].tolist(), fraction=0.02, pad=0.02, label="+CNN helped / −CNN hurt")

    # Row 7: Manually corrected velocity — RdBu_r, tissue masked
    for j, comp in enumerate(COMPONENTS):
        if corr[comp] is not None:
            im = axes[7, j].imshow(
                apply_mask(corr[comp], tissue).T,
                origin="upper", cmap=vel_cmap, vmin=-vel_max, vmax=vel_max,
            )
            axes[7, j].set_title(f"Man. Corr. {comp}")
        else:
            axes[7, j].text(0.5, 0.5, f"Man. Corr. {comp}\nnot available",
                            transform=axes[7, j].transAxes,
                            ha="center", va="center", fontsize=12, color="white")
            axes[7, j].set_title(f"Man. Corr. {comp} (missing)")
    if has_corr:
        fig.colorbar(im, ax=axes[7, :].tolist(), fraction=0.02, pad=0.02, label="velocity")

    # Row 8: Manual polyfit corrected velocity (uncorr + man. polyfit) — RdBu_r, tissue masked
    for j, comp in enumerate(COMPONENTS):
        ax = axes[8, j]
        if has_man_poly_corr:
            im = ax.imshow(
                apply_mask(man_poly_corr[comp], tissue).T,
                origin="upper", cmap=vel_cmap, vmin=-vel_max, vmax=vel_max,
            )
            ax.set_title(f"Man. Poly Corr. {comp}")
        else:
            ax.text(0.5, 0.5, f"Man. Poly Corr. {comp}\nnot available",
                    transform=ax.transAxes,
                    ha="center", va="center", fontsize=12, color="white")
            ax.set_title(f"Man. Poly Corr. {comp} (missing)")
    if has_man_poly_corr:
        fig.colorbar(im, ax=axes[8, :].tolist(), fraction=0.02, pad=0.02, label="velocity")

    # Row 9: Uncorrected velocity — RdBu_r, tissue masked
    for j, comp in enumerate(COMPONENTS):
        if uncorr[comp] is not None:
            im = axes[9, j].imshow(
                apply_mask(uncorr[comp], tissue).T,
                origin="upper", cmap=vel_cmap, vmin=-vel_max, vmax=vel_max,
            )
            axes[9, j].set_title(f"Uncorr. {comp}")
        else:
            axes[9, j].text(0.5, 0.5, f"Uncorr. {comp}\nnot available",
                            transform=axes[9, j].transAxes,
                            ha="center", va="center", fontsize=12, color="white")
            axes[9, j].set_title(f"Uncorr. {comp} (missing)")
    if has_uncorr:
        fig.colorbar(im, ax=axes[9, :].tolist(), fraction=0.02, pad=0.02, label="velocity")

    # Row 10: Magnitude | Mask | Used Pixels
    axes[10, 0].imshow(mag_slice.T, origin="upper", cmap="gray")
    axes[10, 0].set_title("Mag")

    axes[10, 1].imshow(tissue.T.astype(float), origin="upper", cmap="gray")
    axes[10, 1].set_title("Mask")

    axes[10, 2].imshow(apply_mask(mag_slice, tissue).T, origin="upper", cmap="RdBu_r")
    axes[10, 2].set_title("Used Pixels")

    for ax in axes.ravel():
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_facecolor(bg_color)

    return fig

In [18]:
errors = []

for idx, row in patients_df.iterrows():
    pid = row["patient_id"]
    split = row["split"]
    patient_dir = PATIENT_DATA_DIR / pid
    ds_root = patient_dir / "nifti" / DOWNSAMPLED_FOLDER

    if not ds_root.exists():
        print(f"[SKIP] {pid}: downsampled dir not found")
        errors.append((pid, split, "missing_dir"))
        continue

    n_mag_frames = len(list((ds_root / "4d_flow_mag").glob("*.nii.gz")))
    frame = min(FRAME_INDEX, n_mag_frames - 1)

    inf_root = INFERENCE_DIR / pid

    try:
        fig = make_qa_figure(pid, split, frame, ds_root, patient_dir, inf_root=inf_root)
        out_path = OUTPUT_DIR / f"{split}_{pid}.png"
        fig.savefig(out_path, dpi=120, bbox_inches="tight")
        plt.close(fig)
        print(f"[OK]   {pid} ({split}) -> {out_path.name}")
    except Exception as e:
        print(f"[ERR]  {pid} ({split}): {e}")
        errors.append((pid, split, str(e)))

print(f"\nDone. {len(patients_df) - len(errors)} succeeded, {len(errors)} errors.")
if errors:
    print("Errors:")
    for pid, split, msg in errors:
        print(f"  {pid} ({split}): {msg}")

[OK]   Biswifo (test) -> test_Biswifo.png

Done. 1 succeeded, 0 errors.


## Multi-slice deep-dive: Biswifo

In [19]:
DEEP_DIVE_PID = "Biswifo"
DEEP_DIVE_SPLIT = splits_df.loc[splits_df["patient_id"] == DEEP_DIVE_PID, "split"].iloc[0]

patient_dir = PATIENT_DATA_DIR / DEEP_DIVE_PID
ds_root = patient_dir / "nifti" / DOWNSAMPLED_FOLDER
inf_root = INFERENCE_DIR / DEEP_DIVE_PID

n_mag_frames = len(list((ds_root / "4d_flow_mag").glob("*.nii.gz")))
frame = min(FRAME_INDEX, n_mag_frames - 1)

ref_vol = nib.load(str(ds_root / "4d_flow_mag" / f"4d_flow_mag_{DEEP_DIVE_PID}_frame_{frame:02d}.nii.gz"))
n_z = ref_vol.shape[2]

N_SLICES = 16
slice_indices = np.linspace(0, n_z - 1, N_SLICES, dtype=int)

deep_dive_output_dir = OUTPUT_DIR / DEEP_DIVE_PID
deep_dive_output_dir.mkdir(exist_ok=True)

print(f"Patient: {DEEP_DIVE_PID} ({DEEP_DIVE_SPLIT}), frame={frame}, volume z-dim={n_z}")
print(f"Generating {N_SLICES} slices: {slice_indices.tolist()}")

for z in slice_indices:
    fig = make_qa_figure(
        DEEP_DIVE_PID, DEEP_DIVE_SPLIT, frame, ds_root, patient_dir,
        inf_root=inf_root, z_idx=int(z),
    )
    out_path = deep_dive_output_dir / f"z{z:02d}.png"
    fig.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"  [OK] z={z:02d} -> {out_path.name}")

print(f"\nDone. Images saved to {deep_dive_output_dir.resolve()}")

Patient: Biswifo (test), frame=4, volume z-dim=64
Generating 16 slices: [0, 4, 8, 12, 16, 21, 25, 29, 33, 37, 42, 46, 50, 54, 58, 63]
  [OK] z=00 -> z00.png
  [OK] z=04 -> z04.png
  [OK] z=08 -> z08.png
  [OK] z=12 -> z12.png
  [OK] z=16 -> z16.png
  [OK] z=21 -> z21.png
  [OK] z=25 -> z25.png
  [OK] z=29 -> z29.png
  [OK] z=33 -> z33.png
  [OK] z=37 -> z37.png
  [OK] z=42 -> z42.png
  [OK] z=46 -> z46.png
  [OK] z=50 -> z50.png
  [OK] z=54 -> z54.png
  [OK] z=58 -> z58.png
  [OK] z=63 -> z63.png

Done. Images saved to /Users/yakhilesh/Files/7_PhD/vascular-superenhancement/code-base/vascular-superenhancement-4d-flow/notebooks/data-qa/qa_pec_images/lucky-eon_epoch_68/Biswifo
